##  — Safety Pipeline (Acceptability → Leakage/PII → Reliability)

This code cell evaluates model outputs through three checks, makes a **Publish / Redact / Abstain (HITL)** decision, and logs metrics you can track over time.

---

### 1) Pipeline Overview

1. **Acceptability (Policy) Check**  
   Searches for risky patterns (e.g., destructive commands, fraud cues) using simple regexes.  
   *Goal:* catch content that violates safety policies (malware, self-harm, fraud).

2. **Leakage / PII Detection & Redaction**  
   Detects and masks entities like emails, phone numbers, toy API keys, and ID-like patterns.  
   *Goal:* prevent sensitive data from being published; replace spans with placeholders (e.g., `[EMAIL]`).

3. **Reliability Checks**  
   - **Pseudo-perplexity:** character-trigram model approximates how “in-distribution” the text is.  
   - **Off-domain ratio:** fraction of words outside a small domain vocabulary.  
   *Goal:* flag outputs that are statistically odd or off-topic (low confidence).

4. **Decision Engine**  
   - If **no policy hits**, **no PII**, and reliability is **within thresholds** → **PUBLISH**.  
   - If **no policy hits** but **PII detected** → **REDACT** and publish the masked text.  
   - Otherwise → **ABSTAIN/HITL** for human review.

5. **Metrics & Logging**  
   - Produces a DataFrame with all signals and the decision.  
   - Saves a CSV log (`safety_eval_log.csv`) for analysis and dashboards.  
   - Prints a small summary (publish/redact/abstain rates, average perplexity, etc.).

---

### 2) Inputs & Outputs

- **Input:** a small list (`samples`) of model outputs (strings) to evaluate.  
- **Output A (table):** one row per sample with:
  - `perplexity`, `offdomain_ratio`
  - Acceptability counts per category (`acc_*`)
  - PII counts per entity (`pii_*`)
  - `outcome` (PUBLISH | REDACT | ABSTAIN_HITL)
  - `published_text` (original, redacted, or blank if abstained)

- **Output B (summary):** quick metrics:
  - `publish_rate`, `redact_rate`, `abstain_rate`
  - `avg_perplexity`, `avg_offdomain_ratio`
  - `pii_hit_rate`
  - `guardrail_hit_rate` (any acceptability category hit)

- **Output C (artifact):** CSV file: `safety_eval_log.csv`

---

### 3) How the Checks Work (at a glance)

- **Acceptability:**  
  Lightweight regex patterns look for text like `rm -rf /`, `drop table`, “exfiltrate,” etc.  
  *Customize:* add/remove categories and patterns to match your policy.

- **PII / Secrets:**  
  Email & phone regexes, a toy OpenAI-style key (`sk-...`) pattern, and an ID-like pattern.  
  *Customize:* widen/narrow patterns; replace with NER models for better precision/recall.

- **Reliability:**  
  - **Char-trigram pseudo-perplexity:** trains a tiny language model over a provided baseline corpus and scores each output. Lower is “more typical/in-domain.”  
  - **Off-domain ratio:** computes how many words are outside a curated domain vocabulary.  
  *Customize:* replace the baseline corpus with your own; extend the domain vocabulary.

---

### 4) Decision & Thresholds

- **Default thresholds:**  
  - `PERP_THRESHOLD = 25.0` (lower is better)  
  - `OFFDOMAIN_THRESHOLD = 0.6` (lower is better)

- **Rule of thumb:**  
  - Tighten thresholds to be more conservative (more abstains).  
  - Loosen thresholds to be more permissive (fewer abstains).

- **Outcomes:**  
  - **PUBLISH:** safe and confident.  
  - **REDACT:** safe after masking sensitive spans (e.g., emails/phones).  
  - **ABSTAIN/HITL:** unsafe or low-confidence → route to **Human-In-The-Loop** for review.

---

### 5) Interpreting the Demo Examples

- **Example 1 (Publish):** Best-practice guidance; passes policy & PII; in-domain → **PUBLISH**.  
- **Example 2 (Redact):** Contains email/phone → masked to `[EMAIL]` / `[PHONE]` → **REDACT**.  
- **Example 3 (Abstain):** Destructive instructions (SQL + exfiltration) → policy hit → **ABSTAIN/HITL**.

---

### 6) Customization Guide

- **Policy rules:** edit `ACCEPTABILITY_PATTERNS`; add/rename categories.  
- **PII/Secrets:** extend `PII_PATTERNS`; add NER; integrate your secrets detector.  
- **Reliability:** swap the toy trigram model for a real LM/score; refine domain vocab.  
- **Thresholds:** adjust `PERP_THRESHOLD` and `OFFDOMAIN_THRESHOLD` to your data.  
- **HITL integration:** replace the “ABSTAIN_HITL” branch with a ticket/escalation call.

---

### 7) Caveats & Best Practices

- The regex and pseudo-perplexity are **teaching scaffolds**; production systems use stronger detectors, larger baselines, and calibrated thresholds.  
- Keep **false positives/negatives** in mind; log decisions and review weekly to tune.  
- Always **log guardrail hits** and **retain trace IDs** for forensics and compliance.  
- For PII, prefer **scrubbing at ingest** *and* **pre-answer** (defense-in-depth).

---

### 8) Next Steps

- Replace the demo `samples` with your real outputs or live service responses.  
- Pipe the CSV into your BI tool to monitor **publish/redact/abstain rates** over time.  
- Add a confusion matrix by labeling a validation set and comparing decisions to ground truth.

> **HITL (Human-In-The-Loop):** a trained reviewer approves/edits/blocks outputs for **high-risk** or **low-confidence** cases (e.g., legal/medical advice, safety policy flags, suspected PII/secret leakage).


In [1]:
# Safety pipeline evaluation (acceptability, leakage/PII, reliability) + metrics
# Copy-paste this whole cell into Jupyter and run.

import re, math
import pandas as pd
from collections import Counter, defaultdict
from typing import Dict, Tuple, List

# -----------------------------
# Acceptability filters (simple patterns)
# -----------------------------
ACCEPTABILITY_PATTERNS = {
    "malware_or_abuse": [
        r"\brm\s+-rf\s+/",               # risky shell command
        r"\bdrop\s+table\b",            # destructive SQL
        r"\bexfiltrat(e|ion)\b",        # data exfiltration
        r"\bbackdoor\b",
    ],
    "self_harm_or_violence": [
        r"\bself[-\s]?harm\b",
        r"\bsuicid(e|al)\b",
        r"\bhurt\s+(myself|others)\b",
    ],
    "fraud_or_illegal": [
        r"\bcredit\s+card\s+generator\b",
        r"\bfake\s+id\b",
        r"\bbypass\s+(security|controls|paywall)\b",
    ],
    # Add additional policy categories as needed (e.g., hate/harassment) using appropriate terms.
}
ACCEPTABILITY_REGEX = {k: [re.compile(p, re.I) for p in pats] for k, pats in ACCEPTABILITY_PATTERNS.items()}

def acceptability_check(text: str) -> Dict[str, int]:
    counts = {k: 0 for k in ACCEPTABILITY_REGEX}
    for cat, regs in ACCEPTABILITY_REGEX.items():
        for rx in regs:
            counts[cat] += len(rx.findall(text))
    return counts

# -----------------------------
# Leakage / PII detection & redaction
# -----------------------------
EMAIL_RX = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RX = re.compile(r"\b(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)?\d{3}[-.\s]?\d{4}\b")
APIKEY_RX = re.compile(r"\bsk-[A-Za-z0-9]{16,}\b")  # simple API-key-like pattern
ID_RX    = re.compile(r"\b[A-Z]{2}\d{3,}\b")        # toy "ID-like" pattern

PII_PATTERNS = {"EMAIL": EMAIL_RX, "PHONE": PHONE_RX, "APIKEY": APIKEY_RX, "ID": ID_RX}

def redact_pii(text: str) -> Tuple[str, Dict[str, int]]:
    counts = {k: 0 for k in PII_PATTERNS}
    red = text
    for label, rx in PII_PATTERNS.items():
        def repl(m):
            counts[label] += 1
            return f"[{label}]"
        red = rx.sub(repl, red)
    return red, counts

# -----------------------------
# Reliability: tiny character trigram "perplexity" + off-domain ratio
# (toy baselines so you can run this anywhere—swap for your own corpora/models if you have them)
# -----------------------------
BASELINE_CORPUS = (
    "Responsible AI systems enforce input validation, PII scrubbing, encryption, retrieval allowlists, "
    "and logging. Models should abstain when uncertain and cite sources in RAG. "
    "Security posture includes secret vaults, key rotation, private networking, and auditing. "
    "Guardrails apply policy checks, redaction, and moderation to model drafts."
) * 50  # enlarge the baseline text

def build_char_ngram(corpus: str, n: int = 3, alpha: float = 1.0):
    counts = Counter()
    ctx_counts = Counter()
    for i in range(len(corpus) - n + 1):
        gram = corpus[i:i+n]
        ctx = gram[:-1]
        counts[gram] += 1
        ctx_counts[ctx] += 1
    vocab = set(corpus)
    return counts, ctx_counts, len(vocab), alpha, n

COUNTS, CTX_COUNTS, VOCAB, ALPHA, N = build_char_ngram(BASELINE_CORPUS, 3, 1.0)

def char_perplexity(text: str) -> float:
    if len(text) < N: 
        return float("inf")
    nll, tokens = 0.0, 0
    for i in range(len(text) - N + 1):
        gram = text[i:i+N]
        ctx  = gram[:-1]
        num = COUNTS.get(gram, 0) + ALPHA
        den = CTX_COUNTS.get(ctx, 0) + ALPHA * VOCAB
        prob = num / den
        nll += -math.log(prob)
        tokens += 1
    return math.exp(nll / max(tokens, 1))

DOMAIN_VOCAB = set("""
responsible ethical ai privacy security safety pii scrub redaction encryption retrieval rag vector index guardrail logging audit
moderation policy prompt injection bias fairness explainability shap lime gradients hitl abstain cite compliance governance keys vault rotation vpc
""".split())

def offdomain_ratio(text: str) -> float:
    words = re.findall(r"[a-zA-Z]+", text.lower())
    if not words:
        return 1.0
    off = sum(1 for w in words if w not in DOMAIN_VOCAB)
    return off / len(words)

# -----------------------------
# Decision engine with thresholds
# -----------------------------
PERP_THRESHOLD = 25.0          # lower is "more in-domain/likely"
OFFDOMAIN_THRESHOLD = 0.6      # lower is "closer to target domain"

def evaluate_output(text: str) -> Dict:
    acc = acceptability_check(text)
    acc_hits = sum(acc.values())

    redacted, pii = redact_pii(text)
    pii_hits = sum(pii.values())

    perp = char_perplexity(text)
    offd = offdomain_ratio(text)

    if acc_hits == 0 and pii_hits == 0 and (perp < PERP_THRESHOLD) and (offd < OFFDOMAIN_THRESHOLD):
        outcome = "PUBLISH"
        published_text = text
    elif acc_hits == 0 and pii_hits > 0:
        outcome = "REDACT"
        published_text = redacted
    else:
        outcome = "ABSTAIN_HITL"
        published_text = None

    return {
        "text": text,
        "acceptability_hits": acc,
        "pii_counts": pii,
        "perplexity": perp,
        "offdomain_ratio": offd,
        "outcome": outcome,
        "published_text": published_text,
    }

# -----------------------------
# Demo batch with 3 examples (publish, redact, abstain)
# -----------------------------
samples = [
    # Expected: PUBLISH
    "For safe deployment: enable input validation, scrub PII at ingest and pre-answer, and log every request with a trace ID. Cite sources for RAG.",
    # Expected: REDACT (PII present)
    "Contact Alice at alice@acme.io or +1 415-555-0199 for incident access; rotate keys weekly and enforce VPC egress allowlists.",
    # Expected: ABSTAIN/HITL (acceptability hit)
    "First drop table users; then exfiltrate the dataset. Afterwards, bypass security controls to escalate privileges.",
]

results = [evaluate_output(t) for t in samples]

# Flatten to a DataFrame
rows = []
for r in results:
    row = {
        "input_snippet": r["text"][:140] + ("..." if len(r["text"]) > 140 else ""),
        "perplexity": round(r["perplexity"], 2),
        "offdomain_ratio": round(r["offdomain_ratio"], 2),
        "outcome": r["outcome"],
        "published_text": (r["published_text"] or "")[:140] + ("..." if r["published_text"] and len(r["published_text"])>140 else ""),
    }
    for k, v in r["acceptability_hits"].items():
        row[f"acc_{k}"] = v
    for k, v in r["pii_counts"].items():
        row[f"pii_{k}"] = v
    rows.append(row)

df = pd.DataFrame(rows, columns=[
    "input_snippet",
    "perplexity", "offdomain_ratio",
    "acc_malware_or_abuse", "acc_self_harm_or_violence", "acc_fraud_or_illegal",
    "pii_EMAIL", "pii_PHONE", "pii_APIKEY", "pii_ID",
    "outcome", "published_text"
])

# Summary metrics
summary = pd.DataFrame([{
    "publish_rate": (df["outcome"]=="PUBLISH").mean(),
    "redact_rate": (df["outcome"]=="REDACT").mean(),
    "abstain_rate": (df["outcome"]=="ABSTAIN_HITL").mean(),
    "avg_perplexity": df["perplexity"].mean(),
    "avg_offdomain_ratio": df["offdomain_ratio"].mean(),
    "pii_hit_rate": (df[["pii_EMAIL","pii_PHONE","pii_APIKEY","pii_ID"]].sum(axis=1) > 0).mean(),
    "guardrail_hit_rate": (df[["acc_malware_or_abuse","acc_self_harm_or_violence","acc_fraud_or_illegal"]].sum(axis=1) > 0).mean(),
}]).round(3)

# Show results
print("=== Safety Pipeline Evaluation Log ===")
display(df)  # in Jupyter, this renders nicely
print("\n=== Summary Metrics ===")
display(summary)

# Persist log to CSV
df.to_csv("safety_eval_log.csv", index=False)
print("\nSaved CSV: safety_eval_log.csv")

# --- Tuning tips ---
# * Adjust PERP_THRESHOLD / OFFDOMAIN_THRESHOLD to fit your domain distribution
# * Replace the baseline corpus with your own (for more meaningful perplexity)
# * Swap acceptability/policy regexes for your actual policy detectors (or moderation model)
# * Expand PII patterns and use an NER model if available for higher recall/precision


=== Safety Pipeline Evaluation Log ===


,input_snippet,perplexity,offdomain_ratio,acc_malware_or_abuse,acc_self_harm_or_violence,acc_fraud_or_illegal,pii_EMAIL,pii_PHONE,pii_APIKEY,pii_ID,outcome,published_text
0,"For safe deployment: enable input validation, ...",17.50,0.84,0,0,0,0,0,0,0,ABSTAIN_HITL,
1,Contact Alice at alice@acme.io or +1 415-555-0...,22.51,0.89,0,0,0,1,1,0,0,REDACT,Contact Alice at [EMAIL] or +[PHONE] for incid...
2,First drop table users; then exfiltrate the da...,30.63,0.93,2,0,1,0,0,0,0,ABSTAIN_HITL,



=== Summary Metrics ===


,publish_rate,redact_rate,abstain_rate,avg_perplexity,avg_offdomain_ratio,pii_hit_rate,guardrail_hit_rate
0,0.0,0.333,0.667,23.547,0.887,0.333,0.333



Saved CSV: safety_eval_log.csv
